In [1]:
import pandas as pd

df_price = pd.read_csv("prix_day_ahead_fr.csv", index_col="dates")
df_price.index = pd.to_datetime(df_price.index, utc=True)
df_price = df_price.sort_index()

# 1. Ramener tout à l'heure : moyenne des quarts d'heure, quand il y en a
prix_h = df_price["price"].resample("h").mean().dropna()

# 2. Passer en heure de Paris et créer la table jour x heure
p = prix_h.tz_convert("Europe/Paris")
table = pd.DataFrame({"prix": p.values, "jour": p.index.date, "heure": p.index.hour})
matrice = table.pivot_table(index="jour", columns="heure", values="prix")

# 3. Garder uniquement les jours qui ont exactement 24 heures
nb_heures = table.groupby("jour").size()
jours_ok = nb_heures[nb_heures == 24].index
matrice = matrice.loc[jours_ok].dropna(how="any")

print(matrice.shape)
print(matrice.iloc[:3, :6])

(4283, 24)
heure           0      1      2      3      4      5
jour                                                
2015-01-05  36.56  34.94  32.19  23.53  22.92  34.26
2015-01-06  45.56  40.28  39.20  36.22  34.04  38.58
2015-01-07  40.00  36.69  37.45  32.08  29.48  34.94


In [3]:
matrice.index = pd.to_datetime(matrice.index)
matrice = matrice.asfreq("D")  # un index journalier continu, les jours absents deviennent NaN
print(matrice.shape)

baseline_j1 = matrice.shift(1)  # même heure, la veille
baseline_j7 = matrice.shift(7)  # même heure, une semaine avant

debut_test = pd.Timestamp("2025-10-01")

def mae(prevision):
    ecart = (matrice - prevision).loc[debut_test:]
    return ecart.abs().stack().mean()  # moyenne sur tous les jours et toutes les heures

print("MAE baseline J-1 :", round(mae(baseline_j1), 2), "€/MWh")
print("MAE baseline J-7 :", round(mae(baseline_j7), 2), "€/MWh")

(4306, 24)
MAE baseline J-1 : 25.82 €/MWh
MAE baseline J-7 : 37.23 €/MWh


In [4]:
niveau_j1 = matrice.shift(1).mean(axis=1)            # moyenne des 24 prix de D-1
jours = pd.to_datetime(matrice.index)

features = pd.DataFrame(index=matrice.index)
features["niveau_j1"] = niveau_j1
features["jour_semaine"] = jours.dayofweek

# Encodage binaire du jour de la semaine (une colonne par jour, sauf un de référence)
features = pd.get_dummies(features, columns=["jour_semaine"], drop_first=True, dtype=float)

print(features.shape)
print(features.head())

(4306, 7)
            niveau_j1  jour_semaine_1  jour_semaine_2  jour_semaine_3  \
jour                                                                    
2015-01-05        NaN             0.0             0.0             0.0   
2015-01-06  44.426250             1.0             0.0             0.0   
2015-01-07  45.163333             0.0             1.0             0.0   
2015-01-08  45.365833             0.0             0.0             1.0   
2015-01-09  43.014167             0.0             0.0             0.0   

            jour_semaine_4  jour_semaine_5  jour_semaine_6  
jour                                                        
2015-01-05             0.0             0.0             0.0  
2015-01-06             0.0             0.0             0.0  
2015-01-07             0.0             0.0             0.0  
2015-01-08             0.0             0.0             0.0  
2015-01-09             1.0             0.0             0.0  


In [8]:
heure = 12

cible = matrice[heure]                       # prix à 12h du jour D
prix_j1 = matrice.shift(1)[heure]            # prix à 12h de D-1
prix_j7 = matrice.shift(7)[heure]            # prix à 12h de D-7

jeu = features.copy()
jeu["cible"] = cible
jeu["prix_j1"] = prix_j1
jeu["prix_j7"] = prix_j7

# On garde les lignes complètes, et on coupe la période de test
jeu = jeu.dropna()
train = jeu.loc[:"2025-09-30"]
test = jeu.loc["2025-10-01":]

print("train :", train.shape, "| test :", test.shape)
print(jeu.head())
# print(test.isna().sum().sum(), "valeurs manquantes dans le test")

train : (3852, 10) | test : (378, 10)
            niveau_j1  jour_semaine_1  jour_semaine_2  jour_semaine_3  \
jour                                                                    
2015-01-12  26.017500             0.0             0.0             0.0   
2015-01-13  36.273333             1.0             0.0             0.0   
2015-01-14  36.758333             0.0             1.0             0.0   
2015-01-15  39.089167             0.0             0.0             1.0   
2015-01-16  36.850417             0.0             0.0             0.0   

            jour_semaine_4  jour_semaine_5  jour_semaine_6  cible  prix_j1  \
jour                                                                         
2015-01-12             0.0             0.0             0.0  42.49    31.80   
2015-01-13             0.0             0.0             0.0  46.04    42.49   
2015-01-14             0.0             0.0             0.0  45.50    46.04   
2015-01-15             0.0             0.0             0.0  

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

variables = [c for c in jeu.columns if c != "cible"]

modele = LinearRegression()
modele.fit(train[variables], train["cible"])

prevision = modele.predict(test[variables])
mae_modele = mean_absolute_error(test["cible"], prevision)

mae_base = mean_absolute_error(test["cible"], test["prix_j1"])

print("MAE régression :", round(mae_modele, 2))
print("MAE baseline J-1 :", round(mae_base, 2))

MAE régression : 29.16
MAE baseline J-1 : 28.04


In [9]:
#ajout de la météo pour améliorer le modèle

# 1. Météo à 12h heure de Paris, une ligne par jour
meteo = pd.read_csv("meteo_horaire_paris.csv", index_col="time")
meteo.index = pd.to_datetime(meteo.index, utc=True).tz_convert("Europe/Paris")
meteo_12 = meteo[meteo.index.hour == 12].copy()
meteo_12.index = meteo_12.index.normalize().tz_localize(None)  # date naïve, comme la matrice
meteo_12 = meteo_12[["temperature_2m", "wind_speed_100m", "shortwave_radiation", "cloud_cover"]]
meteo_12.columns = [c + "_D" for c in meteo_12.columns]

# 2. Ajout au jeu d'entraînement, puis suppression des lignes incomplètes
jeu_m = jeu.join(meteo_12, how="left")
print("Jours avec météo :", jeu_m[meteo_12.columns[0]].notna().sum(), "sur", len(jeu_m))

jeu_c = jeu_m.dropna()  # jeu commun : mêmes lignes pour tous les modèles
train_c = jeu_c.loc[:"2025-09-30"]
test_c = jeu_c.loc["2025-10-01":]
print("train :", train_c.shape, "| test :", test_c.shape)

# 3. Trois prévisions sur les mêmes lignes de test
vars_sans_meteo = [c for c in jeu.columns if c != "cible"]
vars_avec_meteo = vars_sans_meteo + list(meteo_12.columns)

m0 = LinearRegression().fit(train_c[vars_sans_meteo], train_c["cible"])
m1 = LinearRegression().fit(train_c[vars_avec_meteo], train_c["cible"])

print("MAE baseline J-1 :", round(mean_absolute_error(test_c["cible"], test_c["prix_j1"]), 2))
print("MAE régression sans météo :", round(mean_absolute_error(test_c["cible"], m0.predict(test_c[vars_sans_meteo])), 2))
print("MAE régression avec météo :", round(mean_absolute_error(test_c["cible"], m1.predict(test_c[vars_avec_meteo])), 2))

Jours avec météo : 1724 sur 4230
train : (1348, 14) | test : (376, 14)
MAE baseline J-1 : 27.78
MAE régression sans météo : 29.7
MAE régression avec météo : 28.19


In [ ]:
#ajout de la prévision consommation 

# Consommation prévue (J-1) : moyenne des quarts d'heure de 12h, heure de Paris
df_ec = pd.read_csv("eco2mix_15min.csv", usecols=["date_heure", "prevision_j1"], index_col="date_heure")
df_ec.index = pd.to_datetime(df_ec.index, utc=True).tz_convert("Europe/Paris")
df_ec["prevision_j1"] = pd.to_numeric(df_ec["prevision_j1"], errors="coerce")

# Vérification du pas réel de la colonne
print(df_ec["prevision_j1"].dropna().index.to_series().diff().value_counts().head())

h12 = df_ec[df_ec.index.hour == 12]["prevision_j1"]
conso_prevue = h12.groupby(h12.index.date).mean()
conso_prevue.index = pd.to_datetime(conso_prevue.index)
conso_prevue.name = "conso_prevue_D"

# Jeu commun : mêmes jours pour toutes les comparaisons
jeu_f = jeu_c.join(conso_prevue, how="inner").dropna()
train_f = jeu_f.loc[:"2025-09-30"]
test_f = jeu_f.loc["2025-10-01":]
print("train :", train_f.shape, "| test :", test_f.shape)

vars_meteo = vars_avec_meteo
vars_conso = vars_avec_meteo + ["conso_prevue_D"]

m_meteo = LinearRegression().fit(train_f[vars_meteo], train_f["cible"])
m_conso = LinearRegression().fit(train_f[vars_conso], train_f["cible"])

print("MAE baseline J-1 :", round(mean_absolute_error(test_f["cible"], test_f["prix_j1"]), 2))
print("MAE avec météo :", round(mean_absolute_error(test_f["cible"], m_meteo.predict(test_f[vars_meteo])), 2))
print("MAE avec météo + conso prévue :", round(mean_absolute_error(test_f["cible"], m_conso.predict(test_f[vars_conso])), 2))

date_heure
0 days 00:15:00    518901
0 days 01:15:00        14
Name: count, dtype: int64
train : (1348, 15) | test : (376, 15)
MAE baseline J-1 : 27.78
MAE avec météo : 28.19
MAE avec météo + conso prévue : 28.3
